# Parameterize notebooks with native widgets

Native widgets (`aidputils.widgets`) let you run the same notebook with different input values, interactively or from a job. This notebook covers:

- Defining text, dropdown, combobox, and multiselect widgets.
- Reading widget values interactively and in jobs.
- Overriding widget values from job task parameters and from `aidputils.notebook.run`.
- **Failing fast when a job passes a value a widget does not allow.** If a job passes a value that a dropdown or multiselect does not offer, AIDP keeps the widget's default and does not raise an error, so a scheduled run can process the wrong environment or region without any sign of a problem. Section 3 shows a small check that turns this into a clear error.

**Prerequisites:** an AIDP Workbench notebook attached to a running Spark cluster. No tables or extra libraries are needed.

`aidputils` is available in every notebook; you do not need to import it.

## 1. Define widgets

Each call creates a widget and returns its current value as a string. Widget names must be unique in a notebook.

> **Note:** Re-running a widget call with a different `defaultValue` does not change an existing widget. To change a default, remove the widget first (see section 5).

In [ ]:
W = aidputils.widgets

W.text(name="run_date", defaultValue="2026-01-01", label="Run date")
W.dropdown(name="environment", defaultValue="dev", choices=["dev", "test", "prod"], label="Environment")
W.combobox(name="campaign", defaultValue="win_back", choices=["renewal_outreach", "win_back"], label="Campaign")
W.multiselect(name="regions", defaultValue="North America", choices=["North America", "Europe", "Asia Pacific"], label="Regions")

## 2. Read widget values

`get` returns a string. Multiselect values come back as one comma-separated string, so split them yourself. `getAll` returns every widget value as a dictionary.

In [ ]:
run_date = W.get("run_date")
environment = W.get("environment")
campaign = W.get("campaign")
regions = [r.strip() for r in W.get("regions").split(",") if r.strip()]

print("run_date   :", run_date)
print("environment:", environment)
print("campaign   :", campaign)
print("regions    :", regions)
print("all values :", W.getAll())

## 3. Fail fast on values a widget does not allow

When a job passes a task parameter with the same key as a widget, the parameter overrides the widget value. For a **dropdown** or **multiselect**, a value that is not in `choices` is ignored without an error, and the widget keeps its default. A **combobox** accepts any value.

The helper below compares the raw task parameter with the widget value and raises an error when the override was rejected, so a scheduled job does not run with the wrong value.

> Do not use commas inside multiselect choices. A comma-separated override is split on commas, so a choice such as `"Asia, Pacific"` can never be selected from a parameter.

In [ ]:
_MISSING = "__not_passed__"


def strict_get(name, multiselect=False):
    """Return the widget value; raise if a task parameter for this widget was rejected."""
    value = W.get(name)
    passed = oidlUtils.parameters.getParameter(name, _MISSING)
    if passed == _MISSING:
        return value
    if multiselect:
        if {p.strip() for p in passed.split(",")} != {v.strip() for v in value.split(",")}:
            raise ValueError(f"Parameter {name}={passed!r} was rejected; widget kept {value!r}")
    elif passed != value:
        raise ValueError(f"Parameter {name}={passed!r} was rejected; widget kept {value!r}")
    return value


environment = strict_get("environment")
regions = strict_get("regions", multiselect=True)
print("validated environment:", environment)
print("validated regions    :", regions)

## 4. Pass values to another notebook

`aidputils.notebook.run(path, timeout_seconds, parameters)` runs a child notebook. Keys in `parameters` override the child's widgets with the same names. Whatever the child passes to `aidputils.notebook.exit(value)` is returned as a string.

Set `CHILD_NOTEBOOK` to the absolute path of a notebook that defines a `run_date` widget, then run the cell. It is skipped when the path is empty.

In [ ]:
CHILD_NOTEBOOK = ""  # for example "/Workspace/Shared/child_widgets.ipynb"

if CHILD_NOTEBOOK:
    result = aidputils.notebook.run(CHILD_NOTEBOOK, 300, {"run_date": run_date, "environment": environment})
    print("child returned:", result)
else:
    print("Set CHILD_NOTEBOOK to try notebook-to-notebook parameters.")

## 5. Remove widgets

`remove(name)` deletes one widget and `removeAll()` deletes every widget in the notebook. Remove a widget before redefining it with a new default.

In [ ]:
# W.remove("campaign")
# W.removeAll()

## Run this notebook as a job

1. Create a job with a notebook task that points to this notebook.
2. Add task parameters whose keys match widget names, for example `environment = prod` and `regions = Europe,Asia Pacific`.
3. Run the job. The values from the task parameters replace the widget defaults.
4. To see section 3 in action, set `environment = staging`. The run fails with a clear error instead of silently using `dev`.